# NLP Analysis - Reading Text Files with Regular Expressions

A **Regular Expression (regex)** is a pattern that describes a set of strings. Python's built-in `re` module lets us search, extract, split, and replace text using such patterns. In this notebook, we will read `Numbers_and_Powers.txt` (an essay on Pythagoras and his followers) and perform the text-processing tasks of the *Reading PDFs* notebook using **only the `re` module** instead of `PyPDF2` and `spaCy`.

The exercises include reading a file, splitting it into paragraphs, searching for words, using match objects and groups, extracting dates and names, cleaning text, tokenizing, counting stopwords, approximate lemmatization, and a regex-based approximation of Named Entity Recognition.

This notebook combines the concepts from `5_Regular_Expressions` and `6_Regular_Expressions_2`.

## 1. Importing the Necessary Libraries

Unlike the PDF notebook, nothing needs to be installed here. `re` is part of Python's standard library, and `Counter` is used to count the frequency of words and patterns.

In [1]:
# Import the regular expressions module.
import re

# Import Counter to count the frequency of words and matches.
from collections import Counter

## 2. Read the Text File

### **Exercise 1:** Reading a Text File

In this exercise, we will open `Numbers_and_Powers.txt` with Python's `open()` function. A text file has no pages like a PDF, so we will measure the document in characters, lines, and later in paragraphs.

This is the first step because the file's contents must be loaded into a string before any regular expression can be applied.

In [2]:
# Open the text file and read its entire content into one string.
with open("Numbers_and_Powers.txt", "r", encoding="utf-8") as file:
    full_text = file.read()

# Display the number of characters in the file.
print("Total characters:", len(full_text))

# Display the number of lines in the file.
print("Total lines:", len(full_text.splitlines()))

# Display the beginning of the text to verify that it was read successfully.
print()
print(full_text[:500])

Total characters: 8803
Total lines: 172

PART 1. PYTHAGORAS, HIs TENETS AND HIs
FOLLOWERS

Pythagoras, one of the greatest philosophers of ancient Europe, was the
son of Mnesarchus, an engraver. He was born about the year 580 B.C.,
either at Samos, an island in the Agean Sea, or, as some Say, at Sidon in
Phoenicia. Very little is known of his early life, beyond the fact that he won
prizes for feats of agility at the Olympic Games. Having attained manhood
and feeling dissatisfied with the amount of knowledge to be gained at
home, he lef


## 3. Split the text into paragraphs

### **Exercise 2:** Split the Text into Paragraphs with `re.split()`

A PDF is divided into pages; this text file is divided into paragraphs separated by blank lines. `re.split()` splits a string wherever a pattern matches.

The pattern `\n\s*\n` means: a newline, followed by any whitespace, followed by another newline, which is a blank line. The resulting list `paragraphs` plays the same role as `page_texts` in the PDF notebook.

In [3]:
# Split the text wherever one or more blank lines occur.
paragraphs = re.split(r"\n\s*\n", full_text)

# Remove empty strings and strip surrounding whitespace.
paragraphs = [p.strip() for p in paragraphs if p.strip()]

# Display the number of paragraphs.
print("Total paragraphs:", len(paragraphs))

Total paragraphs: 34


## 4. Read a specific paragraph

### **Exercise 3:** Read the First Paragraph and a Specific Paragraph

Just as we selected pages by index in the PDF notebook, we can select paragraphs by index. Python uses zero-based indexing, so index `0` is the first paragraph.

In [4]:
# Display the first paragraph.
print("First paragraph:\n")
print(paragraphs[0])

# Specify the paragraph number we want to read (this example reads paragraph 4).
paragraph_number = 4

# Check that the requested paragraph exists.
if 1 <= paragraph_number <= len(paragraphs):

    # Display the selected paragraph using its zero-based index.
    print(f"\nParagraph {paragraph_number}:\n")
    print(paragraphs[paragraph_number - 1])

else:
    # Display an error message if the number is invalid.
    print("The requested paragraph does not exist.")

First paragraph:

PART 1. PYTHAGORAS, HIs TENETS AND HIs
FOLLOWERS

Paragraph 4:

Among the ancient authors from whom we derive our knowledge of the life
and doctrines of Pythagoras and his successors, the following are notable:—


## 5. Search for a pattern with `re.search()`

### **Exercise 4:** Find a Pattern and Inspect the Match Object

`re.search()` scans the string and returns a **match object** for the *first* place where the pattern occurs, or `None` if it does not occur.

A match object tells us what was matched and where:

- `match.group()` is the matched text.
- `match.span()` is the `(start, end)` positions.
- `match.start()` and `match.end()` give each position separately.

In [5]:
# Search for the first occurrence of "Pythagoras".
match = re.search("Pythagoras", full_text)

# Check whether the pattern was found.
if match:
    print("Pattern found")
    print("Matched text:", match.group())
    print("Span        :", match.span())
    print("Start       :", match.start())
    print("End         :", match.end())
else:
    print("Pattern not found")

# A pattern that does not occur returns None.
print("\nSearch for 'Napoleon':", re.search("Napoleon", full_text))

Pattern found
Matched text: Pythagoras
Span        : (50, 60)
Start       : 50
End         : 60

Search for 'Napoleon': None


**Observation:**

`re.search()` stops at the first match, so it cannot tell us how many times a word occurs. The next exercises use `re.findall()` and `re.finditer()` for that.

## 6. Search for a word in every paragraph

### **Exercise 5:** Search for a Word or Phrase

In this exercise, we will search for the word *number* in every paragraph. Passing `re.IGNORECASE` makes the search case-insensitive, so `number`, `Number`, and `NUMBER` are treated alike.

In [6]:
# Specify the word to search for.
search_word = "number"

# Create a list to store the paragraph numbers containing the word.
matching_paragraphs = []

# Search through every paragraph.
for paragraph_number, paragraph in enumerate(paragraphs, start=1):

    # re.escape() makes sure special characters in the word are matched literally.
    if re.search(re.escape(search_word), paragraph, re.IGNORECASE):

        # Store the paragraph number.
        matching_paragraphs.append(paragraph_number)

# Display the result.
if matching_paragraphs:
    print("Found in paragraphs:", matching_paragraphs)
else:
    print("The search term was not found in the text.")

Found in paragraphs: [10, 14, 15, 23, 25, 26, 31, 33]


## 7. All occurrences: `findall()` and `finditer()`

### **Exercise 6:** Find Every Occurrence and Its Position

- `re.findall()` returns a list of all matching strings.
- `re.finditer()` returns one match object per occurrence, so we also get the positions.

In [7]:
# Return all occurrences of "Pythagoras" as a list of strings.
all_matches = re.findall("Pythagoras", full_text)
print("Occurrences of 'Pythagoras':", len(all_matches))

# Return match objects so that we can display the span of each occurrence.
print("\nPositions of the first 5 occurrences:")
for match in list(re.finditer("Pythagoras", full_text))[:5]:
    print(match.span())

Occurrences of 'Pythagoras': 12

Positions of the first 5 occurrences:
(50, 60)
(1508, 1518)
(2230, 2240)
(2676, 2686)
(3978, 3988)


## 8. Count the words

### **Exercise 7:** Count Words in the Document

In the PDF notebook, words were counted with `split()`. Here we compare that estimate with a regular expression. The pattern `\w+` matches one or more word characters (letters, digits, and underscore), so punctuation does not become part of a word.

In [8]:
# Count words using split() (whitespace-separated pieces).
split_count = len(full_text.split())

# Count words using the regular expression \w+ .
words = re.findall(r"\w+", full_text)
regex_count = len(words)

# Display both counts.
print("Word count using split():", split_count)
print("Word count using \\w+   :", regex_count)

# Display the first 20 words found.
print("\nFirst 20 words:", words[:20])

Word count using split(): 1414
Word count using \w+   : 1446

First 20 words: ['PART', '1', 'PYTHAGORAS', 'HIs', 'TENETS', 'AND', 'HIs', 'FOLLOWERS', 'Pythagoras', 'one', 'of', 'the', 'greatest', 'philosophers', 'of', 'ancient', 'Europe', 'was', 'the', 'son']


**Observation:**

The counts differ because `split()` treats `A.D.—Nicomachus` or `"autos` as one piece, while `\w+` separates the words from the punctuation attached to them.

## 9. Save the extracted text to files

### **Exercise 8:** Save the Paragraphs to a New Text File

As in the PDF notebook, we will save the processed text to a new file. We label each paragraph so that the boundaries remain identifiable. We will also save only the first paragraph as a separate file, similar to copying the first page into a new PDF.

In [9]:
# Save all paragraphs with a heading before each one.
with open("Numbers_and_Powers_Extracted_Text.txt", "w", encoding="utf-8") as output_file:
    for paragraph_number, paragraph in enumerate(paragraphs, start=1):
        output_file.write(f"\n--- Paragraph {paragraph_number} ---\n")
        output_file.write(paragraph)

print("Extracted text saved as Numbers_and_Powers_Extracted_Text.txt")

# Save only the first paragraph in a separate file.
with open("Numbers_and_Powers_First_Paragraph.txt", "w", encoding="utf-8") as output_file:
    output_file.write(paragraphs[0])

print("First paragraph saved as Numbers_and_Powers_First_Paragraph.txt")

Extracted text saved as Numbers_and_Powers_Extracted_Text.txt
First paragraph saved as Numbers_and_Powers_First_Paragraph.txt


## 10. Character classes

### **Exercise 9:** Shorthand Character Classes

Regular expressions provide shorthand codes for common groups of characters:

| Code | Meaning |
|------|---------|
| `\d` | a digit |
| `\D` | a non-digit |
| `\w` | a letter, digit or underscore |
| `\W` | a non-alphanumeric character |
| `\s` | a whitespace character |
| `\S` | a non-whitespace character |

In this exercise, we will count how many characters of each class occur in the text.

In [10]:
# Define each shorthand class and a description.
classes = {
    r"\d": "digits",
    r"\D": "non-digits",
    r"\w": "alphanumeric characters",
    r"\W": "non-alphanumeric characters",
    r"\s": "whitespace characters",
    r"\S": "non-whitespace characters",
}

# Count the matches of each class in the full text.
for code_, meaning in classes.items():
    print(f"{code_}  {meaning:<30}: {len(re.findall(code_, full_text))}")

\d  digits                        : 62
\D  non-digits                    : 8741
\w  alphanumeric characters       : 7023
\W  non-alphanumeric characters   : 1780
\s  whitespace characters         : 1452
\S  non-whitespace characters     : 7351


## 11. Extract dates and years

### **Exercise 10:** Extract Years and Dates

The text gives years in several formats: `580 B.C.`, `B.C. 450`, `50 A.D.`, `300 A. D.` and plain four-digit years such as `1620`. We will write one pattern for each format.

- `\b\d{4}\b` matches a four-digit number surrounded by word boundaries (`\b`).
- `\.` matches a literal full stop, because an unescaped `.` is a wildcard.
- `A\.\s?D\.` allows an optional space inside `A. D.`

In [11]:
# Years written as "<number> B.C." (for example, 580 B.C.).
number_then_bc = re.findall(r"\b\d+\s*B\.C\.", full_text)

# Years written as "B.C. <number>" (for example, B.C. 450).
bc_then_number = re.findall(r"B\.C\.\s*\d+", full_text)

# Years written as "<number> A.D." or "<number> A. D.".
ad_years = re.findall(r"\b\d+\s*A\.\s?D\.", full_text)

# Plain four-digit years (for example, 1620).
plain_years = re.findall(r"\b\d{4}\b", full_text)

print("<number> B.C. :", number_then_bc)
print("B.C. <number> :", bc_then_number)
print("A.D. years    :", ad_years)
print("Four-digit    :", plain_years)

<number> B.C. : ['580 B.C.', '500 B.C.']
B.C. <number> : ['B.C. 450', 'B.C. 394', 'B.C. 380', 'B.C. 370', 'B.C. 322', 'B.C. 276', 'B.C. 40']
A.D. years    : ['50 A.D.', '300 A.D.', '340 A.D.', '450 A.D.', '560 A.D.', '850 A.D.', '300 A. D.']
Four-digit    : ['1620', '1650', '1660']


## 12. Groups and compiled patterns

### **Exercise 11:** Capture Parts of a Match with Groups

Parentheses `( )` create **groups**, which capture parts of a match. `re.compile()` stores a pattern in a reusable object, which is useful when the same pattern is used many times.

Here we will capture the era and the number from every date, then use `group(1)`, `group(2)`, and so on to read each part. Named groups `(?P<name>...)` make the result easier to read.

In [12]:
# Compile a pattern with named groups for dates written like "B.C. 450".
bc_pattern = re.compile(r"(?P<era>B\.C\.)\s*(?P<year>\d+)")

# Inspect the first match in detail.
first = bc_pattern.search(full_text)
print("Entire match :", first.group())
print("Group 1 (era):", first.group(1))
print("Group 2 (yr) :", first.group(2))
print("As a dict    :", first.groupdict())

# Collect every (era, year) pair with findall().
print("\nAll B.C. dates as (era, year):")
for era, year in bc_pattern.findall(full_text):
    print(f"  {year} {era}")

Entire match : B.C. 450
Group 1 (era): B.C.
Group 2 (yr) : 450
As a dict    : {'era': 'B.C.', 'year': '450'}

All B.C. dates as (era, year):
  450 B.C.
  394 B.C.
  380 B.C.
  370 B.C.
  322 B.C.
  276 B.C.
  40 B.C.


## 13. The OR operator and the wildcard

### **Exercise 12:** Alternation (`|`) and the Wildcard (`.`)

- The OR operator `|` matches either of two patterns, for example `man|woman`.
- The wildcard `.` matches any single character except a newline.
- `\S+` matches one or more non-whitespace characters, which is useful for matching whole words.

The Pythagoreans named geometric objects after numbers. We use `|` to find those names, and the wildcard to find spelling variations of *Pythagoras*.

In [13]:
# Use | to find the Pythagorean number names (case-insensitive).
number_names = re.findall(r"monad|dyad|triad|tetrad", full_text, re.IGNORECASE)
print("Number names:", Counter(n.lower() for n in number_names))

# Use the wildcard . to match "Pythagoras", "Pythagorean", "Pythagorica", etc.
variants = re.findall(r"P.thagor\w+", full_text)
print("\nPythagoras variants:", Counter(variants))

# Use \S+ to find whole words ending with "tion".
tion_words = re.findall(r"\S+tion\b", full_text)
print("\nWords ending with 'tion':", sorted(set(tion_words)))

Number names: Counter({'monad': 2, 'dyad': 2, 'triad': 2, 'tetrad': 2})

Pythagoras variants: Counter({'Pythagoras': 12, 'Pythagorean': 3, 'Pythagoreans': 2, 'Pythagorica': 1})

Words ending with 'tion': ['Imagination', 'Sensation', 'Tradition', 'affection', 'causation', 'consideration', 'contemplation', 'evolution', 'information', 'institution', 'instruction', 'motion', 'opposition', 'revelation', 'selection']


## 14. Starts with and ends with

### **Exercise 13:** Anchors `^` and `$`

- `^` matches the **start** of a string and `$` matches the **end**.
- By default these refer to the start and end of the *whole* string. Passing `re.MULTILINE` makes them match at the start and end of *every line*.

The text contains a list of ancient authors, where each line starts with a date. We will use anchors to pick those lines.

In [14]:
# Lines that start with "B.C." (authors dated before Christ).
bc_lines = re.findall(r"^B\.C\..*$", full_text, re.MULTILINE)
print("Lines starting with 'B.C.':", len(bc_lines))
for line in bc_lines[:3]:
    print("  ", line)

# Lines that start with a number (authors dated with A.D.).
number_lines = re.findall(r"^\d+.*$", full_text, re.MULTILINE)
print("\nLines starting with a number:", len(number_lines))
for line in number_lines[:3]:
    print("  ", line)

# Lines that end with a colon followed by a dash, which introduce the lists.
intro_lines = re.findall(r"^.*:\u2014$", full_text, re.MULTILINE)
print("\nLines ending with ':\u2014':")
for line in intro_lines:
    print("  ", line)

# Without MULTILINE, ^ and $ apply to the whole string.
print("\nWhole text starts with 'PART':", bool(re.search(r"^PART", full_text)))
print("Whole text ends with a full stop:", bool(re.search(r"\.\s*$", full_text)))

Lines starting with 'B.C.': 7
   B.C. 450.—Herodotus, who speaks of the mysteries of the Pythagoreans as
   B.C. 394.—Archytas of Tarentum, who left a fragment upon Pythagorean
   B.C. 380.—Theon of Smyrna.

Lines starting with a number: 6
   50 A.D.—Nicomachus of Gerasa; Treatises on Arithmetic and Harmony.
   300 A.D.—Porphyry of Tyre, a great philosopher, sometimes named in
   340 A.D.—Jamblicus wrote "De mysteriis," "De vita Pythagorica," "The

Lines ending with ':—':
   and doctrines of Pythagoras and his successors, the following are notable:—

Whole text starts with 'PART': True
Whole text ends with a full stop: True


## 15. Excluding characters

### **Exercise 14:** Exclude Characters with `[^...]`

Placing `^` at the start of a bracket set **excludes** the characters inside it. For example, `[^\d]` matches anything that is not a digit and `[^!.?, ]+` matches runs of characters that are not punctuation or spaces.

We will use this to strip the punctuation from the first sentence.

In [15]:
# Take the first sentence of the second paragraph.
sentence = re.split(r"(?<=[.!?])\s+", paragraphs[1])[0]
print("Original:", sentence)

# Keep only runs of characters that are not punctuation or whitespace.
clean_words = re.findall(r"[^!.?,;:\s]+", sentence)
print("\nWords   :", clean_words)

# Join the words back together with single spaces.
print("Cleaned :", " ".join(clean_words))

# Exclude digits: everything that is not a digit, in runs.
print("\nText around digits:", re.findall(r"[^\d]+", "Born 580 B.C., died 500 B.C."))

Original: Pythagoras, one of the greatest philosophers of ancient Europe, was the
son of Mnesarchus, an engraver.

Words   : ['Pythagoras', 'one', 'of', 'the', 'greatest', 'philosophers', 'of', 'ancient', 'Europe', 'was', 'the', 'son', 'of', 'Mnesarchus', 'an', 'engraver']
Cleaned : Pythagoras one of the greatest philosophers of ancient Europe was the son of Mnesarchus an engraver

Text around digits: ['Born ', ' B.C., died ', ' B.C.']


## 16. Hyphenated words

### **Exercise 15:** Find Hyphenated Words

A word joined by a hyphen can be found with `\w+-\w+`: one or more word characters, a hyphen, and one or more word characters.

In [16]:
# Find all hyphenated words.
hyphenated = re.findall(r"\w+-\w+", full_text)

print("Hyphenated words:", sorted(set(hyphenated)))
print("Total occurrences:", len(hyphenated))

Hyphenated words: ['by-word', 'ever-famous']
Total occurrences: 2


## 17. Compiled patterns with groups

### **Exercise 16:** Extract Names and Places

The list of ancient authors contains phrases such as *Archytas of Tarentum* and *Theon of Smyrna*. The pattern `([A-Z]\w+) of ([A-Z]\w+)` captures the person in group 1 and the place in group 2.

The later authors are listed as `Surname, Firstname, Year`. A second pattern captures all three parts.

In [17]:
# Compile a pattern for "<Person> of <Place>".
person_of_place = re.compile(r"([A-Z]\w+) of ([A-Z]\w+)")

print("People and their places:\n")
for person, place in person_of_place.findall(full_text):
    print(f"  {person:<12} -> {place}")

# Compile a pattern for "<Surname>, <Firstname>, <Year>".
surname_pattern = re.compile(r"([A-Z]\w+), ([A-Z]\w+), (\d{4})")

print("\nRecent authors (surname, first name, year):\n")
for surname, first_name, year in surname_pattern.findall(full_text):
    print(f"  {first_name} {surname} ({year})")

People and their places:

  Brahmins     -> India
  Hierophants  -> Egypt
  Oracle       -> Delphi
  Archytas     -> Tarentum
  Theon        -> Smyrna
  Nicomachus   -> Stagyra
  Nicomachus   -> Gerasa
  Porphyry     -> Tyre
  Arithmetic   -> Nicomachus
  Simplicius   -> Cilicia
  Photius      -> Constantinople

Recent authors (surname, first name, year):

  Johannes Meursius (1620)
  Marcus Meibomius (1650)
  Athanasius Kircher (1660)


**Observation:**

The first pattern also matches phrases that merely look like `<Capitalized> of <Capitalized>`, such as `Oracle of Delphi` (a place, not a person) and `Arithmetic of Nicomachus` (a book title). A pattern only checks the shape of the text, so the results still need to be read by a person.

## 18. Build a timeline of ancient authors

### **Exercise 17:** Combine Anchors, Groups, and OR

We will combine what we have learned to extract each author in the chronological list at the start of a line, together with the date.

- `(?P<date>B\.C\. \d+|\d+ A\.D\.)` uses `|` to accept both date formats.
- `\.\u2014` matches the full stop and em dash that follow the date.
- `(?P<name>[^,.;]+)` captures the name up to the first comma, full stop, or semicolon.

In [18]:
# Compile a pattern that captures the date and the author name.
timeline_pattern = re.compile(
    r"^(?P<date>B\.C\. \d+|\d+ A\.D\.)\.\u2014(?P<name>[^,.;]+)",
    re.MULTILINE
)

# Collect the dates and names.
timeline = [(m.group("date"), m.group("name").strip()) for m in timeline_pattern.finditer(full_text)]

print("Timeline of authors:\n")
for date, name in timeline:
    print(f"{date:<12} {name}")

print("\nTotal authors:", len(timeline))

Timeline of authors:

B.C. 450     Herodotus
B.C. 394     Archytas of Tarentum
B.C. 380     Theon of Smyrna
B.C. 370     Philolaus
B.C. 322     Aristotle
B.C. 276     Eratosthenes
B.C. 40      Cicero

Total authors: 7


## 19. Extract quoted text

### **Exercise 18:** Extract Quotations

Quotations are enclosed in double quotation marks. The pattern `"([^"]+)"` matches an opening quote, then captures everything that is not a quote, up to the closing quote.

Because this file was converted from a scan, some quotation marks are curly or mismatched, so a plain `"([^"]+)"` pairs the wrong marks. To reduce this, we use lookarounds: an **opening** quote must be preceded by whitespace (or the start) and followed by a non-space, and a **closing** quote must be preceded by a non-space and must not be followed by a letter or digit.

In [19]:
# Simple version: everything between two straight double quotes.
simple_quotes = re.findall(r'"([^"]+)"', full_text)

# Improved version using lookbehind (?<=...) and lookahead (?=...).
quote_pattern = re.compile(r'(?<!\S)"(?=\S)([^"]+?)(?<=\S)"(?!\w)')
quotes = quote_pattern.findall(full_text)

print("Simple pattern  :", len(simple_quotes), "quotations")
print("Improved pattern:", len(quotes), "quotations")
print()
for quote in quotes:
    # Replace line breaks inside a quotation with a space for readability.
    print("-", re.sub(r"\s+", " ", quote))

Simple pattern  : 22 quotations
Improved pattern: 20 quotations

- Metaphysica,
- Nicomachean Ethics.
- Kokkinon
- Cribrum,
- De Finibus
- De natura Deorum.
- De mysteriis,
- De vita Pythagorica,
- The Arithmetic of Nicomachus of Gerasa,
- The Theological Properties of Numbers.
- Works and Days
- autos epha
- my friend is my other self,
- Italian school,
- Doric school.
- The numerals of Pythagoras,
- were hieroglyphic symbols, by means whereof he explained all ideas concerning the nature of things,
- Secret Doctrine,
- Numbers are a key to the ancient views of cosmogony—in its broad sense, spiritually as well as physically considered, and to the evolution of the present human race; all systems of religious mysticism are based upon numerals. The sacredness of numbers begins with the Great First Cause, the One, and ends only with the nought or zero—symbol of the infinite and boundless universe.
- Isis Unveiled,


**Observation:**

The improved pattern is much cleaner, but it still misses a quotation such as `a"Sieve to separate...` where the scan removed the space before the opening mark. A regular expression only follows the characters it is given, so imperfect source text always limits the result.

## 20. Extract words starting with a specific letter

### **Exercise 19:** Words Starting with a Specific Letter

`\b` marks a word boundary. The pattern `\b[Pp]\w+` matches words that begin with an upper- or lower-case *p*.

In [20]:
# Find words that start with P or p.
p_words = re.findall(r"\b[Pp]\w+", full_text)

print("Total words starting with P/p:", len(p_words))
print("\nMost common:")
for word, count in Counter(w.lower() for w in p_words).most_common(10):
    print(f"  {word:<15} {count}")

# Find capitalised words that are not at the beginning of a sentence.
capitalised = re.findall(r"(?<=[a-z,] )[A-Z][a-z]+", full_text)
print("\nUnique capitalised words inside sentences (first 30):")
print(sorted(set(capitalised))[:30])

Total words starting with P/p: 74

Most common:
  pythagoras      13
  pupils          3
  pythagorean     3
  principles      3
  philosophers    2
  philosopher     2
  pythagoreans    2
  philolaus       2
  porphyry        2
  pupil           2

Unique capitalised words inside sentences (first 30):
['Agean', 'Archytas', 'Aristzeus', 'Arithmetic', 'Asia', 'Aspendian', 'Athanasius', 'Auscultantes', 'Bibliotheca', 'Brahmins', 'Bulagoras', 'Cause', 'Central', 'Chaldean', 'Cilicia', 'Clinias', 'College', 'Composite', 'Constantinople', 'Crete', 'Crotona', 'Days', 'Delphi', 'Deorum', 'Diodorus', 'Doctrine', 'Dorians', 'Dyad', 'Egypt', 'Esoteric']


## 21. Case-insensitive matching

### **Exercise 20:** Compare Case-Sensitive and Case-Insensitive Matching

By default, regular expressions are case-sensitive. The flag `re.IGNORECASE` (or `re.I`) ignores case. The optional `s?` in `numbers?` matches both the singular and the plural.

In [21]:
# Case-sensitive search.
sensitive = re.findall(r"numbers?", full_text)

# Case-insensitive search.
insensitive = re.findall(r"numbers?", full_text, re.IGNORECASE)

print("Case-sensitive   :", len(sensitive), Counter(sensitive))
print("Case-insensitive :", len(insensitive), Counter(insensitive))

Case-sensitive   : 5 Counter({'numbers': 4, 'number': 1})
Case-insensitive : 12 Counter({'Numbers': 7, 'numbers': 4, 'number': 1})


## 22. Searching for emails, phone numbers, URLs, hashtags and mentions

### **Exercise 21:** Common Extraction Patterns

The earlier regex notebooks used patterns for emails, phone numbers, URLs, hashtags, and mentions. We will apply the same patterns to this text.

| Item | Pattern |
|------|---------|
| Email | `[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}` |
| Phone (10 digits) | `\b\d{10}\b` |
| URL | `https?://\S+` |
| Hashtag | `#\w+` |
| Mention | `(?<!\w)@\w+` |

The `#` and `@` symbols must be part of the pattern. A plain `\w+` would match every word, not only hashtags or mentions. The lookbehind `(?<!\w)` stops the mention pattern from matching the `@gmail` part of an email address.

In [22]:
# Define the patterns in a dictionary.
patterns = {
    "Emails":   r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    "Phones":   r"\b\d{10}\b",
    "URLs":     r"https?://\S+",
    "Hashtags": r"#\w+",
    "Mentions": r"(?<!\w)@\w+",
}

# A short sample string used to confirm that every pattern works.
sample = "Mail rahul@gmail.com or call 9087654321. See https://www.python.org #Python @Neha"

print(f"{'Type':<10}{'In Numbers_and_Powers.txt':<30}In sample string")
print("-" * 70)
for name, pattern in patterns.items():
    in_file = re.findall(pattern, full_text)
    in_sample = re.findall(pattern, sample)
    print(f"{name:<10}{str(in_file):<30}{in_sample}")

Type      In Numbers_and_Powers.txt     In sample string
----------------------------------------------------------------------
Emails    []                            ['rahul@gmail.com']
Phones    []                            ['9087654321']
URLs      []                            ['https://www.python.org']
Hashtags  []                            ['#Python']
Mentions  []                            ['@Neha']


**Observation:**

This text is an essay on ancient philosophy, so it contains none of these items. The sample string confirms that the patterns themselves work; an empty list is the correct answer for the file.

## 23. Replace and clean text with `re.sub()`

### **Exercise 22:** Cleaning the Text

`re.sub(pattern, replacement, string)` replaces every match. We will use it to clean the text and to fix a few errors caused by the scan, such as `ina`, `toa`, and `HIs`.

- `[^\w\s]` removes punctuation.
- `\s+` replaced with a single space removes extra whitespace.
- Backreferences such as `\2` and `\1` in the replacement reuse captured groups.

In [23]:
# Take a sample of the text to keep the output short.
sample_text = " ".join(re.split(r"(?<=[.!?])\s+", paragraphs[1])[:2])

# 1. Fix OCR errors: "HIs" -> "His", "ina" -> "in a", "toa" -> "to a".
fixed = re.sub(r"\bHIs\b", "His", full_text)
fixed = re.sub(r"\bina\b", "in a", fixed)
fixed = re.sub(r"\btoa\b", "to a", fixed)
print("Occurrences of 'ina'/'toa'/'HIs' replaced:", len(re.findall(r"\b(?:ina|toa|HIs)\b", full_text)))
print("Before:", re.search(r".{20}\bina\b.{12}", full_text, re.S).group().replace("\n", " "))
print("After :", re.search(r".{20}\bin a\b.{12}", fixed, re.S).group().replace("\n", " "))

# 2. Remove punctuation.
no_punct = re.sub(r"[^\w\s]", "", sample_text)
print("\nWithout punctuation:\n", no_punct)

# 3. Collapse extra spaces and line breaks into a single space.
single_spaced = re.sub(r"\s+", " ", no_punct)
print("\nWith extra spaces removed:\n", single_spaced)

# 4. Use a backreference to reorder "B.C. 450" into "450 BC".
print("\nReformatted dates:", re.sub(r"B\.C\.\s*(\d+)", r"\1 BC", "B.C. 450.\u2014Herodotus; B.C. 40.\u2014Cicero"))

# 5. Replace every occurrence of a word.
print("\nReplace words:", re.sub(r"Pythagoras", "Pythagoras of Samos", "Pythagoras was a philosopher. Pythagoras taught numbers."))

Occurrences of 'ina'/'toa'/'HIs' replaced: 4
Before: y. The teaching was ina great measu
After : y. The teaching was in a great measu

Without punctuation:
 Pythagoras one of the greatest philosophers of ancient Europe was the
son of Mnesarchus an engraver He was born about the year 580 BC
either at Samos an island in the Agean Sea or as some Say at Sidon in
Phoenicia

With extra spaces removed:
 Pythagoras one of the greatest philosophers of ancient Europe was the son of Mnesarchus an engraver He was born about the year 580 BC either at Samos an island in the Agean Sea or as some Say at Sidon in Phoenicia

Reformatted dates: 450 BC.—Herodotus; 40 BC.—Cicero

Replace words: Pythagoras of Samos was a philosopher. Pythagoras of Samos taught numbers.


## 24. Prepare the text for analysis

### **Exercise 23:** Tokenization and Sentences

The PDF notebook used spaCy to split text into tokens and sentences. With regular expressions:

- Tokens: `\w+|[^\w\s]` matches either a word or a single punctuation mark.
- Sentences: `re.split(r"(?<=[.!?])\s+", text)` splits after `.`, `!` or `?` followed by whitespace. The `(?<=...)` part is a **lookbehind**, meaning the punctuation is checked but not removed.

Both methods are approximations. A regex split would, for example, break `B.C. 450` incorrectly, but it is quick and good enough for an overview.

In [24]:
# Tokenize: each word or punctuation mark becomes one token.
tokens = re.findall(r"\w+|[^\w\s]", full_text)

# Split into sentences after ., ! or ? followed by whitespace.
sentences = re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", full_text))

# Keep alphabetic words only (no digits or underscores).
alpha_words = re.findall(r"\b[A-Za-z]+\b", full_text)

print("Total tokens   :", len(tokens))
print("Alphabetic words:", len(alpha_words))
print("Total sentences:", len(sentences))

print("\nFirst tokens   :", tokens[:15])
print("\nFirst sentence :", sentences[0])
print("Second sentence:", sentences[1])

Total tokens   : 1774
Alphabetic words: 1425
Total sentences: 64

First tokens   : ['PART', '1', '.', 'PYTHAGORAS', ',', 'HIs', 'TENETS', 'AND', 'HIs', 'FOLLOWERS', 'Pythagoras', ',', 'one', 'of', 'the']

First sentence : PART 1.
Second sentence: PYTHAGORAS, HIs TENETS AND HIs FOLLOWERS Pythagoras, one of the greatest philosophers of ancient Europe, was the son of Mnesarchus, an engraver.


## 25. Identify stopwords

### **Exercise 24:** Identify Stopwords

Stopwords are common words such as `the`, `is`, `and`, `of`, and `to`. spaCy has a built-in list; with `re` we define our own list and match tokens against it.

The pattern is built with `"|".join(...)` and anchored with `^...$`, so that only whole words match.

In [25]:
# Define a list of common English stopwords.
stopwords = [
    "the", "a", "an", "and", "or", "but", "of", "to", "in", "on", "at", "by", "for", "with",
    "from", "as", "is", "are", "was", "were", "be", "been", "it", "its", "he", "his", "him",
    "she", "her", "they", "their", "them", "we", "our", "us", "i", "my", "me", "you", "your",
    "that", "this", "these", "those", "which", "who", "whom", "had", "has", "have", "not",
    "so", "than", "then", "there", "into", "also", "one", "all", "any", "each", "no", "who",
]

# Build a regex from the list: ^(the|a|an|...)$
stopword_regex = re.compile(r"^(?:" + "|".join(stopwords) + r")$", re.IGNORECASE)

# Select the alphabetic words that are stopwords.
stopword_list = [w.lower() for w in alpha_words if stopword_regex.match(w)]

# Count the occurrences of each stopword.
stopword_counts = Counter(stopword_list)

print("Most frequent stopwords:\n")
for word, count in stopword_counts.most_common(15):
    print(word, ":", count)

print("\nTotal stopword occurrences:", len(stopword_list))
print("Unique stopwords:", len(stopword_counts))

Most frequent stopwords:

the : 112
of : 100
and : 65
to : 41
his : 29
a : 28
was : 23
in : 22
he : 16
that : 14
or : 11
from : 11
by : 11
as : 10
is : 10

Total stopword occurrences: 636
Unique stopwords: 51


**Observation:**

Whether stopwords should be removed depends on the task. For example, removing them helps find the topic words of a text, but it can damage tasks that depend on grammar.

## 26. Most frequent content words

### **Exercise 25:** Word Frequency Without Stopwords

After removing stopwords and very short words, the remaining words describe what the document is about.

In [26]:
# Keep words with 4 or more letters that are not stopwords.
content_words = [
    w.lower() for w in alpha_words
    if len(w) >= 4 and not stopword_regex.match(w)
]

content_counts = Counter(content_words)

print("Most frequent content words:\n")
for word, count in content_counts.most_common(15):
    print(f"{word:<15} {count}")

Most frequent content words:

pythagoras      13
numbers         11
knowledge       7
school          7
great           5
secret          5
doctrines       5
first           5
followers       4
known           4
most            4
well            4
upon            4
arithmetic      4
ideas           4


## 27. Pattern matching for themes

### **Exercise 26:** Pattern Matching with Named Patterns

In the PDF notebook, spaCy's `Matcher` was used to register several named patterns and find them all. We can do the same with a dictionary of regular expressions. Unlike `Matcher`, a regex works on characters rather than on tokens or parts of speech, so we use `\b` and optional endings such as `s?` to control the boundaries.

In [27]:
# Define named patterns for the main themes of the text.
theme_patterns = {
    "PYTHAGORAS":  r"\bPythagoras\b",
    "PYTHAGOREAN": r"\bPythagorean(?:s)?\b",
    "NUMBER":      r"\bnumbers?\b",
    "SCHOOL":      r"\b(?:school|college|society)\b",
    "MYSTERY":     r"\b(?:secret|mystery|mysteries|esoteric|occult)\b",
    "PHILOSOPHER": r"\bphilosoph(?:er|ers|y)\b",
    "GEOMETRY":    r"\b(?:point|line|superficies|solid|geometry|magnitude)\b",
}

# Run each pattern over the document and keep the matched text.
theme_matches = {}
for name, pattern in theme_patterns.items():
    theme_matches[name] = re.findall(pattern, full_text, re.IGNORECASE)

# Display the matches found for each pattern.
print("Matched patterns:\n")
for name, found in theme_matches.items():
    print(f"{name:<12} {len(found):>3}  {dict(Counter(f.lower() for f in found))}")

print("\nTotal pattern matches:", sum(len(f) for f in theme_matches.values()))

Matched patterns:

PYTHAGORAS    13  {'pythagoras': 13}
PYTHAGOREAN    5  {'pythagoreans': 2, 'pythagorean': 3}
NUMBER        12  {'numbers': 11, 'number': 1}
SCHOOL        10  {'college': 2, 'society': 1, 'school': 7}
MYSTERY        9  {'esoteric': 1, 'secret': 5, 'occult': 1, 'mysteries': 2}
PHILOSOPHER    5  {'philosophers': 2, 'philosopher': 2, 'philosophy': 1}
GEOMETRY       8  {'magnitude': 3, 'geometry': 1, 'point': 1, 'line': 1, 'superficies': 1, 'solid': 1}

Total pattern matches: 62


**Observation:**

The pattern `\bPythagorean(?:s)?\b` matches `Pythagorean` and `Pythagoreans` but not `Pythagorica`. A regular expression finds only what its pattern spells out; it does not understand meaning.

## 28. Approximate lemmatization

### **Exercise 27:** Simple Suffix Stripping

Lemmatization converts a word to its dictionary form (`memories` -> `memory`). spaCy uses a trained model. With `re.sub()` we can only approximate it with a few suffix rules:

- `ies` -> `y` (`memories` -> `memory`)
- `sses` or `uses` -> `ss` or `us` (`losses` -> `loss`)
- a final `s` is removed unless it follows `a`, `i`, `o`, `s` or `u` (`metals` -> `metal`, but `Pythagoras`, `this`, and `Samos` are protected)
- words of three letters or fewer are left alone (`was`, `his`)

This is called **stemming by rules**. It cannot handle irregular words such as `children` -> `child`, and it may wrongly change words like `analysis`.

In [28]:
def simple_lemma(word):
    """Apply simple regex suffix rules to approximate the base form of a plural noun."""
    word = word.lower()
    if len(word) <= 3:
        return word                                   # leave short words such as "was" alone
    word = re.sub(r"ies$", "y", word)                 # memories -> memory
    word = re.sub(r"(ss|us)es$", r"\1", word)         # losses -> loss
    word = re.sub(r"(?<=[^aiosu])s$", "", word)       # numbers -> number
    return word

# Create (word, approximate lemma) pairs for alphabetic words.
lemma_results = [(w, simple_lemma(w)) for w in alpha_words]

print("Original word -> Approximate lemma\n")
for word, lemma in lemma_results[:20]:
    print(word, "->", lemma)

print("\nAlphabetic tokens processed:", len(lemma_results))

Original word -> Approximate lemma

PART -> part
PYTHAGORAS -> pythagoras
HIs -> his
TENETS -> tenet
AND -> and
HIs -> his
FOLLOWERS -> follower
Pythagoras -> pythagoras
one -> one
of -> of
the -> the
greatest -> greatest
philosophers -> philosopher
of -> of
ancient -> ancient
Europe -> europe
was -> was
the -> the
son -> son
of -> of

Alphabetic tokens processed: 1425


## 29. Find words whose form changes

### **Exercise 28:** Words That Change After Lemmatization

As with the noun exercise in the PDF notebook, we list the unique words whose form changes after applying the rules above. Words of three letters or fewer are skipped so that short words such as `his` and `has` are not shortened.

In [29]:
# Collect unique (word, lemma) pairs where the form changed.
changed_words = []
seen = set()

for word, lemma in lemma_results:
    key = (word.lower(), lemma)
    if len(word) > 3 and word.lower() != lemma and key not in seen:
        seen.add(key)
        changed_words.append((word, lemma))

print("Words that change after lemmatization (first 40):\n")
for word, lemma in changed_words[:40]:
    print(f"{word} -> {lemma}")

print("\nUnique changed word forms:", len(changed_words))

Words that change after lemmatization (first 40):

TENETS -> tenet
FOLLOWERS -> follower
philosophers -> philosopher
prizes -> prize
feats -> feat
Games -> game
years -> year
centres -> centre
narrates -> narrate
stores -> store
Polycrates -> polycrate
Dorians -> dorian
Students -> student
Gymnosophists -> gymnosophist
Brahmins -> brahmin
Hierophants -> hierophant
pupils -> pupil
powers -> power
inhabitants -> inhabitant
authors -> author
doctrines -> doctrine
successors -> successor
speaks -> speak
mysteries -> mystery
Pythagoreans -> pythagorean
books -> book
Ethics -> ethic
Eratosthenes -> eratosthene
Numbers -> number
works -> work
Treatises -> treatise
sometimes -> sometime
Properties -> property
Days -> day
gives -> give
views -> view
times -> time
Johannes -> johanne
instructors -> instructor
Theorides -> theoride

Unique changed word forms: 83


**Observation:**

The rules handle regular plurals well (`prizes` -> `prize`, `mysteries` -> `mystery`, `Treatises` -> `treatise`), but they also change words that are not plurals, such as the names `Eratosthenes` -> `eratosthene` and `Johannes` -> `johanne`. They would also miss irregular forms like `children`. This shows why a trained lemmatizer is better than regular expressions for this task.

## 30. Identify auxiliary verbs

### **Exercise 29:** Find Auxiliary Verbs

spaCy tags auxiliary verbs automatically. With `re`, we list the common auxiliary verbs and count them. This works because auxiliary verbs form a small, closed set of words.

In [30]:
# Compile a pattern matching common auxiliary verbs.
aux_pattern = re.compile(
    r"\b(?:is|are|was|were|be|been|being|am|has|have|had|do|does|did|"
    r"will|would|shall|should|can|could|may|might|must)\b",
    re.IGNORECASE
)

auxiliary_verbs = aux_pattern.findall(full_text)
auxiliary_verb_counts = Counter(v.lower() for v in auxiliary_verbs)

print("Auxiliary verb frequencies:\n")
for verb, count in auxiliary_verb_counts.most_common():
    print(verb, ":", count)

print("\nTotal auxiliary verb occurrences:", len(auxiliary_verbs))

Auxiliary verb frequencies:

was : 23
is : 10
are : 7
were : 7
be : 4
had : 4
have : 2
has : 2
should : 1
did : 1
been : 1
must : 1
being : 1

Total auxiliary verb occurrences: 64


**Observation**

Regular expressions cannot tell whether `had` is an auxiliary (`had gathered`) or a main verb (`had a college`). spaCy decides this from the sentence, but `re` counts every occurrence.

## 31. Find adjectives by their endings

### **Exercise 30:** Words with Adjective Endings

Without a part-of-speech tagger, adjectives can only be guessed from common endings such as `-ous`, `-ive`, `-ic`, `-ical`, `-al`, `-ful`, `-able`, and `-ible`. We also keep the sentence in which each word occurs so it can be judged in context.

In [31]:
# Compile a pattern for words that end like adjectives (at least 3 letters before the suffix).
adjective_pattern = re.compile(
    r"\b[A-Za-z]{3,}(?:ous|ive|ical|ic|ful|able|ible)\b"
    r"|\b[A-Za-z]{4,}al\b"
)

# Collect each candidate with its sentence.
adjective_results = []
seen = set()
for sentence in sentences:
    for match in adjective_pattern.finditer(sentence):
        word = match.group()
        if word.lower() not in seen:
            seen.add(word.lower())
            adjective_results.append((word, sentence))

print("Possible adjectives (first 20):\n")
for word, sentence in adjective_results[:20]:
    print("Word   :", word)
    print("Context:", sentence[:120] + ("..." if len(sentence) > 120 else ""))
    print("-" * 70)

print("Unique possible adjectives:", len(adjective_results))

Possible adjectives (first 20):

Word   : Olympic
Context: Very little is known of his early life, beyond the fact that he won prizes for feats of agility at the Olympic Games.
----------------------------------------------------------------------
Word   : native
Context: Having attained manhood and feeling dissatisfied with the amount of knowledge to be gained at home, he left his native l...
----------------------------------------------------------------------
Word   : Esoteric
Context: History narrates that his pilgrimage in search of wisdom extended to Egypt, Hindostan, Persia, Crete and Palestine, and ...
----------------------------------------------------------------------
Word   : exoteric
Context: History narrates that his pilgrimage in search of wisdom extended to Egypt, Hindostan, Persia, Crete and Palestine, and ...
----------------------------------------------------------------------
Word   : impracticable
Context: He returned with his mind well stored and his judgment m

**Observation**

This method guesses by spelling, so it includes words that only look like adjectives (the noun `Arithmetic` and the verb `derive`) and misses adjectives with other endings (for example `great`). The surrounding sentence must be read to judge the real meaning.

## 32. Recognize dates and quantities

### **Exercise 31:** Identify Dates and Numeric Values

The PDF notebook extracted `DATE` and `MONEY` entities with spaCy. Here we collect every date expression with one combined regular expression (using `|` and `(?: )` for non-capturing groups), and search for monetary values separately.

In [32]:
# One pattern for all date styles found in this text.
date_pattern = re.compile(
    r"(?:about the year )?"
    r"(?:B\.C\.\s*\d+|\d+\s*B\.C\.|\d+\s*A\.\s?D\.|\b1[0-9]{3}\b)"
)

date_entities = [m.group().strip() for m in date_pattern.finditer(full_text)]

print("Dates found:\n")
for date in date_entities:
    print("DATE:", date)

print("\nTotal DATE entities:", len(date_entities))

# Search for money: a currency symbol followed by a number.
money = re.findall(r"[$\u00a3\u20ac\u20b9]\s?\d[\d,]*(?:\.\d+)?", full_text)
print("MONEY entities:", money)

# Count spelled-out durations such as "five years" or "forty years".
durations = re.findall(
    r"\b(?:one|two|three|four|five|six|seven|eight|nine|ten|twenty|thirty|forty|fifty|many)\s+years?\b",
    full_text
)
print("\nDurations:", durations)

Dates found:

DATE: about the year 580 B.C.
DATE: about the year 500 B.C.
DATE: B.C. 450
DATE: B.C. 394
DATE: B.C. 380
DATE: B.C. 370
DATE: B.C. 322
DATE: B.C. 276
DATE: B.C. 40
DATE: 50 A.D.
DATE: 300 A.D.
DATE: 340 A.D.
DATE: 450 A.D.
DATE: 560 A.D.
DATE: 850 A.D.
DATE: 1620
DATE: 1650
DATE: 1660
DATE: 300 A. D.

Total DATE entities: 19
MONEY entities: []

Durations: ['many years', 'forty years', 'five years']


**Observation**

No monetary values occur in this essay, so the money list is empty. The date pattern depends on the formats we wrote, and a date in a new format would be missed.

## 33. Approximate Named Entity Recognition

### **Exercise 32:** Find People, Places and Works with Capitalization Patterns

spaCy's Named Entity Recognition uses a trained model. A regular expression has no such knowledge, but proper names in English are capitalized, so a pattern for **capitalized words that do not start a sentence** gives a useful rough list.

- `(?<![.!?]\s)` is a negative lookbehind that excludes words directly after a sentence end.
- `[A-Z][a-z]+(?:\s(?:of\s)?[A-Z][a-z]+)*` matches one capitalized word, optionally followed by more capitalized words (and the word `of`), so `Nicomachus of Gerasa` is kept together.

In [33]:
# Flatten the text so that line breaks do not split names.
flat_text = re.sub(r"\s+", " ", full_text)

# Compile a pattern for sequences of capitalized words.
entity_pattern = re.compile(r"(?<![.!?] )\b[A-Z][a-z]+(?:\s(?:of\s)?[A-Z][a-z]+)*")

candidates = entity_pattern.findall(flat_text)

# Common capitalized words that are not names.
not_names = {"The", "He", "It", "They", "This", "These", "Among", "No", "So", "Hence", "Intuitive",
             "Reason", "Imagination", "Sensation", "Refer", "Indeed", "Failing", "Having", "Every",
             "Very", "Coming", "Tradition", "We", "Numbers", "Real", "Mathematical", "Great", "Part",
             "Followers", "Tenets", "Hence", "Intense", "History", "Italian", "Doric"}

entities = [c for c in candidates if c not in not_names]
entity_counts = Counter(entities)

print("Most frequent capitalized names:\n")
for name, count in entity_counts.most_common(20):
    print(f"{name:<25} {count}")

print("\nTotal candidate occurrences:", len(entities))
print("Unique candidates:", len(entity_counts))

Most frequent capitalized names:

Pythagoras                12
Europe                    3
De                        3
Mnesarchus                2
College                   2
Crotona                   2
Metapontum                2
Pythagoreans              2
Philolaus                 2
Jamblicus                 2
Pythagorean               2
Samos                     1
Agean Sea                 1
Say                       1
Sidon                     1
Phoenicia                 1
Olympic Games             1
Learning                  1
Egypt                     1
Hindostan                 1

Total candidate occurrences: 142
Unique candidates: 119


**Observation**

This approach finds most names but gives no labels (person, place, organization). It also keeps capitalized words that are not names, such as `Say` (a scan error for `say`) and `De` (from the Latin titles). For labelled entities, a trained NER model such as spaCy is more reliable.

## 34. Highlight a pattern in context

### **Exercise 33:** Show Matches in Context (a simple visualisation)

spaCy's `displaCy` highlights entities in the original text. We can do something similar with `re.sub()` by wrapping every match in square brackets, so we can check each match in its context.

In [34]:
# Take a manageable portion of the text.
portion = full_text[:900]

# Wrap every date in [[ ]] and every number name in << >>.
highlighted = re.sub(r"(B\.C\.\s*\d+|\d+\s*B\.C\.)", r"[[\1]]", portion)
highlighted = re.sub(r"\b(Pythagoras|Pythagorean\w*)\b", r"<<\1>>", highlighted)

print(highlighted)

PART 1. PYTHAGORAS, HIs TENETS AND HIs
FOLLOWERS

<<Pythagoras>>, one of the greatest philosophers of ancient Europe, was the
son of Mnesarchus, an engraver. He was born about the year [[580 B.C.]],
either at Samos, an island in the Agean Sea, or, as some Say, at Sidon in
Phoenicia. Very little is known of his early life, beyond the fact that he won
prizes for feats of agility at the Olympic Games. Having attained manhood
and feeling dissatisfied with the amount of knowledge to be gained at
home, he left his native land and spent many years in travel, visiting in turn
most of the great centres of Learning. History narrates that his pilgrimage in
search of wisdom extended to Egypt, Hindostan, Persia, Crete and Palestine,
and that he gathered from each country fresh stores of information, and
succeeded in becoming well acquainted with the Esoteric Wisdom as well as
with the popular exoteric knowle


**Observation**

Showing the matches in context makes it easy to see whether a pattern is too broad or too narrow before relying on its results.

## 35. Display a final analysis summary

### **Exercise 34:** Summarize the Analysis

In this final exercise, we will display a summary of the file and the regular expression tasks performed in the notebook.

In [35]:
print("=" * 50)
print("NUMBERS AND POWERS: REGEX ANALYSIS SUMMARY")
print("=" * 50)

# Basic file statistics.
print("Total characters:", len(full_text))
print("Total lines:", len(full_text.splitlines()))
print("Total paragraphs:", len(paragraphs))
print("Word count (\\w+):", regex_count)

# Basic text statistics.
print("Total tokens:", len(tokens))
print("Total sentences:", len(sentences))

# Stopword statistics.
print("Stopword occurrences:", len(stopword_list))
print("Unique stopwords:", len(stopword_counts))

# Lemmatization statistics.
print("Alphabetic tokens analyzed:", len(lemma_results))
print("Unique changed word forms:", len(changed_words))

# Grammar-related statistics.
print("Auxiliary verb occurrences:", len(auxiliary_verbs))
print("Possible adjectives:", len(adjective_results))

# Pattern statistics.
print("Theme pattern matches:", sum(len(f) for f in theme_matches.values()))
print("Authors in the timeline:", len(timeline))
print("DATE entities:", len(date_entities))
print("Capitalized name candidates:", len(entities))

print("\nText reading and regex analysis completed.")

NUMBERS AND POWERS: REGEX ANALYSIS SUMMARY
Total characters: 8803
Total lines: 172
Total paragraphs: 34
Word count (\w+): 1446
Total tokens: 1774
Total sentences: 64
Stopword occurrences: 636
Unique stopwords: 51
Alphabetic tokens analyzed: 1425
Unique changed word forms: 83
Auxiliary verb occurrences: 64
Possible adjectives: 32
Theme pattern matches: 62
Authors in the timeline: 7
DATE entities: 19
Capitalized name candidates: 142

Text reading and regex analysis completed.


## 36. Conclusion

### **Conclusion**

In this notebook, we read `Numbers_and_Powers.txt` and analyzed it using only Python's `re` module.

The file exercises showed how to read a text file, split it into paragraphs with `re.split()`, read specific paragraphs, search for words, count words, and save the cleaned text to new files.

The regular expression exercises covered `re.search()`, match objects (`span`, `start`, `end`, `group`), `re.findall()`, `re.finditer()`, `re.compile()`, groups and named groups, the OR operator, the wildcard, the anchors `^` and `$`, excluded character sets `[^...]`, shorthand classes (`\d`, `\w`, `\s` and their opposites), word boundaries, lookbehinds, flags such as `IGNORECASE` and `MULTILINE`, and `re.sub()` with backreferences.

We then rebuilt the NLP tasks of the PDF notebook with regular expressions: tokenization, stopword counting, pattern matching, approximate lemmatization, auxiliary verbs, adjectives, dates, and a rough form of named entity recognition.

Regular expressions are fast and precise when a pattern is well defined, such as dates, codes, and fixed word lists. They cannot understand grammar or meaning, so tasks like true lemmatization, part-of-speech tagging, and labelled entity recognition are better handled by an NLP library such as spaCy.

---